<a href="https://colab.research.google.com/github/the-rks221/AI-RETROSYNTHETIC-ANALYST/blob/main/AI_Retrosynthetic_Analyst.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%pip install rdkit pubchempy langchain-core langchain-google-genai ipywidgets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.1/38.1 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 76.8 MB/s eta 0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled google-genai-2.12.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.

In [4]:
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown
import pubchempy as pcp
from rdkit import Chem
from rdkit.Chem import Draw, rdChemReactions
import os
import base64
import json
import re

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ==============================================================================
# STEP 1: PASTE YOUR GEMINI API KEY HERE
# ==============================================================================
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')

# Model Selection - Use gemini-1.5-flash or gemini-2.0-flash
MODEL_NAME = "gemini-3.6-flash"

# --- UI Setup ---
title = widgets.HTML(value="<h2 style='color: #FF5733;'>🧪 AI Retrosynthetic Analyst with 2D Diagrams</h2>")
text_input = widgets.Text(value='', description='Target Name:', style={'description_width': 'initial'})
button = widgets.Button(description='Analyze Synthesis', button_style='danger', icon='cogs')
output = widgets.Output()

def extract_text_content(raw_response):
    """Safely extracts string text whether response is string, dict, or list of blocks."""
    if isinstance(raw_response, str):
        return raw_response
    elif isinstance(raw_response, list):
        parts = []
        for block in raw_response:
            if isinstance(block, dict) and 'text' in block:
                parts.append(block['text'])
            else:
                parts.append(str(block))
        return "\n".join(parts)
    return str(raw_response)

def draw_2d_reaction_schemes(json_block):
    """Parses SMILES from JSON and renders 2D pictorial reaction diagrams using RDKit."""
    try:
        steps = json.loads(json_block)
        for step_info in steps:
            step_num = step_info.get("step", 1)
            title_text = step_info.get("title", f"Step {step_num}")
            reactants = step_info.get("reactants", [])
            products = step_info.get("products", [])

            if reactants and products:
                r_smiles = ".".join(reactants)
                p_smiles = ".".join(products)
                rxn_smarts = f"{r_smiles}>>{p_smiles}"

                rxn = rdChemReactions.ReactionFromSmarts(rxn_smarts, useSmiles=True)
                if rxn:
                    display(HTML(f"<h4>🖼️ 2D Pictorial Diagram — Step {step_num}: {title_text}</h4>"))
                    img = Draw.ReactionToImage(rxn)
                    display(img)
    except Exception as e:
        print(f"⚠️ Could not generate 2D reaction diagram: {e}")

def get_retrosynthesis_plan(molecule_name, smiles):
    """Generates the retrosynthesis plan and structured 2D SMILES JSON via Gemini."""
    try:
        llm = ChatGoogleGenerativeAI(model=MODEL_NAME, temperature=0.1)
    except Exception as e:
        return f"Error initializing Gemini LLM: {str(e)}", "[]"

    template = """
    You are an expert synthetic organic chemist. Analyze the following target molecule:
    Name: {molecule}
    SMILES: {smiles}

    Please provide a highly detailed report with the following sections:

    ### 1. Retrosynthetic Analysis
    - Step-by-step disconnections and logic.

    ### 2. Forward Synthesis Scheme
    - Detailed reaction steps with reagents, solvents, and reaction conditions.

    ### 3. Synthetic Equivalents & Market Availability
    - Commercial starting materials, synthons, and costs.

    ### 4. Commercial Feasibility Score (Out of 10)
    - Rating based on cost, step count, green chemistry, and yield.

    CRITICAL: At the VERY END of your response, output a single JSON block formatted exactly like this:
    ```json
    [
      {{
        "step": 1,
        "title": "Nitration of Phenol",
        "reactants": ["Oc1ccccc1"],
        "products": ["Oc1ccc([N+](=O)[O-])cc1"]
      }}
    ]
    ```
    """

    prompt = PromptTemplate(input_variables=["molecule", "smiles"], template=template)

    # StrOutputParser guarantees string output
    chain = prompt | llm | StrOutputParser()

    raw_result = chain.invoke({"molecule": molecule_name, "smiles": smiles})
    text_result = extract_text_content(raw_result)

    # Extract JSON block for 2D diagram generation
    json_str = "[]"
    json_match = re.search(r'```json\s*([\s\S]*?)\s*```', text_result)
    if json_match:
        json_str = json_match.group(1).strip()
        # Remove JSON code block from visible report text for cleaner reading
        text_result = text_result.replace(json_match.group(0), "").strip()

    return text_result, json_str

def create_download_link(report_content, filename):
    """Generates an HTML button to download the report."""
    b64 = base64.b64encode(report_content.encode()).decode()
    return f"""
    <a href="data:text/markdown;base64,{b64}" download="{filename}"
       style="display: inline-block; padding: 10px 18px; margin-top: 15px;
              background-color: #28a745; color: white; text-decoration: none;
              font-weight: bold; border-radius: 6px; font-family: sans-serif;">
       📥 Download Complete Synthesis Report (.md)
    </a>
    """

def on_click(b):
    with output:
        output.clear_output()
        target_name = text_input.value.strip()
        if not target_name:
            print("⚠️ Please enter a target molecule name.")
            return

        print(f"🔍 Fetching structural data for '{target_name}'...")

        try:
            # 1. Fetch PubChem Data
            compounds = pcp.get_compounds(target_name, 'name')
            if not compounds:
                print("❌ Molecule not found in PubChem.")
                return

            target = compounds[0]
            smiles = target.smiles if hasattr(target, 'smiles') else target.isomeric_smiles

            display(HTML(f"<h3>Target: {target.iupac_name}</h3>"))
            display(HTML(f"<b>SMILES:</b> <code>{smiles}</code>"))

            # 2. Draw Target Molecule
            mol = Chem.MolFromSmiles(smiles)
            if mol:
                display(HTML("<h4>Target 2D Structure:</h4>"))
                img = Draw.MolToImage(mol, size=(280, 280))
                display(img)

            # 3. Generate Gemini Synthesis Plan
            print("\n⚙️ Running Gemini retrosynthetic logic engine...")
            report_text, reaction_json = get_retrosynthesis_plan(target_name, smiles)

            # 4. Render Main Report
            display(Markdown(report_text))

            # 5. Render 2D Pictorial Reaction Schemes
            draw_2d_reaction_schemes(reaction_json)

            # 6. Render Download Link
            safe_filename = "".join([c for c in target_name if c.isalnum() or c==' ']).strip().replace(' ', '_')
            download_html = create_download_link(report_text, f"{safe_filename}_Synthesis_Report.md")
            display(HTML(download_html))

        except Exception as e:
            print(f"❌ An error occurred: {e}")

button.on_click(on_click)

# Render UI
display(widgets.VBox([title, text_input, button, output]))